<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/05_subagents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Subagents: delegation and context isolation

Back to the research agent. Look at a long run in Studio and you will see the problem: by step
15, every search result the agent has ever seen is still being re-sent on every model call.

Subagents fix that — but not for the reason people usually assume. A subagent is **a context
boundary first and a specialist second.**

**New in this lesson:** `SubAgent`, the `task` tool, per-subagent tools and models

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-05-subagents"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Baseline: one agent doing everything

A single agent, one broad research question. Run it and watch the message list grow.

In [ ]:
from deepagents import create_deep_agent

solo_agent = create_deep_agent(
    model=MODEL,
    tools=[{"type": "web_search"}],
    system_prompt="You are a thorough research assistant. Cite sources.",
)
solo_agent

In [ ]:
from langsmith_studio_nb import start_studio

start_studio("solo_agent")

Example prompt:
> Compare LangGraph, Deep Agents, and plain LangChain agents across three dimensions: control flow, state management, and when each is the right choice. Research each one before answering.

In Studio, expand the last model call and look at how much is in its input. Every search result
from every earlier step is still there — and was re-sent every single step.

---

## 2. The same task, delegated

A `SubAgent` is a dict: a name, a description of when to use it, and its own system prompt. It
may also override its own tools and model.

The parent gets a `task` tool. When it delegates, the subagent runs its own full agent loop in a
**separate context**, and returns only its final message.

In [ ]:
researcher = {
    "name": "researcher",
    "description": (
        "Researches ONE specific topic in depth using web search and returns a dense summary "
        "with citations. Use this for each topic you need to investigate, one call per topic."
    ),
    "system_prompt": (
        "You research exactly one topic thoroughly using web search.\n"
        "Return a dense, factual summary of at most 300 words with inline citations.\n"
        "Do not pad it with caveats or restate the question."
    ),
    "tools": [{"type": "web_search"}],
}

supervisor = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You coordinate research. For a question spanning multiple topics, delegate each "
        "topic to the `researcher` subagent, then synthesise their summaries into one answer. "
        "Do not research topics yourself."
    ),
    subagents=[researcher],
)
supervisor

In [ ]:
start_studio("supervisor")

Example prompt:
> Compare LangGraph, Deep Agents, and plain LangChain agents across three dimensions: control flow, state management, and when each is the right choice. Research each one before answering.

Send the **same** prompt as before, then find a `task` call in Studio and expand the child run
underneath it.

The parent received **one message**: the subagent's final summary, a few hundred tokens. It
never saw the subagent's own work — every `web_search` call, every page of raw results, every
intermediate step. Those existed only inside the child run and were discarded when it returned.

Three searches returning 8,000 tokens each cost the parent 24,000 tokens forever in the solo
version. Delegated, they cost one 300-word summary.

**The corollary is a real cost:** if the parent later needs a detail the subagent saw but did not
include, that detail is gone. The return message is the only channel — which is why a subagent's
prompt should specify what to *return*, not just what to do.

---

## 3. Each subagent is configured independently

Cheap model for bulk gathering, stronger model for judgement. Narrow tools for narrow jobs.

In [ ]:
gatherer = {
    "name": "gatherer",
    "description": "Collects raw facts on a topic. Fast and cheap. Use for straightforward lookups.",
    "system_prompt": "Collect facts on the topic. Return bullet points with citations. No analysis.",
    "tools": [{"type": "web_search"}],
    "model": MODEL,  # swap for a smaller, cheaper model in production
}

analyst = {
    "name": "analyst",
    "description": (
        "Analyses already-gathered facts and forms a judgement. Has NO search tools — "
        "pass it the facts in your request."
    ),
    "system_prompt": (
        "You reason about facts you are given. State a clear conclusion and name the "
        "strongest argument against it. You cannot search; work with what you are given."
    ),
    "tools": [],  # deliberately toolless: it cannot wander off and browse
}

two_tier = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "Use `gatherer` to collect facts, then pass those facts to `analyst` for judgement. "
        "Synthesise the result."
    ),
    subagents=[gatherer, analyst],
)
two_tier

In [ ]:
start_studio("two_tier")

Example prompt:
> Should a team building a customer support bot use Deep Agents or write a LangGraph graph by hand?

Giving `analyst` **no tools** is a design choice, not an oversight. A subagent with search
available will search, even when you wanted it to think about what it already has.

A subagent's tool list is a constraint you impose on purpose. Removing a tool is a design
decision as real as adding one.

---

## 4. When *not* to delegate

Delegation is not free. Each `task` call is a whole extra agent loop: more latency, more tokens,
and one more place for the intent to get garbled.

Do not delegate when:

- **The task is short.** A single lookup does not need its own context.
- **The subagent would need the full conversation.** Anything requiring nuance from earlier turns
  is better done inline — you would have to re-explain it all anyway.
- **Latency matters.** A user waiting on a chat reply feels every nested loop.
- **The output is hard to summarise.** If you need everything the subagent saw, isolation is
  working against you.

Delegate when the work is **bulky, separable, and summarisable**. Research is the canonical fit.

You have had a general-purpose subagent since lesson 01, by the way — that is why every agent
so far has had a `task` tool without you configuring one.

---

## 📌 Key takeaways

- A subagent is a **context boundary** first and a specialist second.
- The parent sees only the subagent's final message — everything else stays in the child run.
- That isolation cuts both ways: detail the subagent did not summarise is gone for good.
- Each subagent gets its own prompt, tools, and model — a toolless subagent is often deliberate.
- `task` is just a tool, and a general-purpose subagent has been present since lesson 01.
- Delegation costs latency and tokens. Delegate work that is bulky, separable, and summarisable.

---

## ➡️ Next

**[06 · Middleware and human-in-the-loop](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/06_middleware_and_hitl.ipynb)**

You have changed behaviour by changing prompts and tools. Next: changing it at the **seams** —
retries, summarization, PII redaction, and stopping the agent for human approval before it
issues a refund.